# Building Damage Classification

**Exercise:** [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32/cnn-building-damage/blob/main/building_damage-exercise.ipynb)
**Solution:** [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32/cnn-building-damage/blob/main/building_damage.ipynb)


This notebook demonstrates building damage classification using PyTorch and transfer learning.

**Dataset:** PEER Hub ImageNet Φ-Net Building Collapse Mode

**Classes:**
- GC (Global Collapse)
- NC (No Collapse)
- PC (Partial Collapse)

## 1. Import Required Libraries

In [3]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
import random
import shutil
from pathlib import Path
import urllib.request
import zipfile

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image
from tqdm.notebook import tqdm
import time
import copy

# For evaluation
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import pandas as pd

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"MPS (Apple Silicon) available: {torch.backends.mps.is_available()}")

# Determine best device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using device: MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

AttributeError: partially initialized module 'torch' has no attribute 'fx' (most likely due to a circular import)

## 2. Download and Prepare Dataset

**Two options available:**
1. **Hugging Face** (Recommended) - Faster, includes pre-trained model
2. **PEER Hub** (Original) - Direct from source

Choose your preferred method below.

In [ ]:
#  Download from PEER Hub (Original source)
# Use this if you prefer the original source or if Hugging Face is unavailable

def download_dataset(output_dir='data'):
    """Download the PEER Hub Φ-Net dataset"""
    print("Downloading PEER Hub ImageNet Φ-Net Data for Building Collapse Mode...")

    url = 'https://apps.peer.berkeley.edu/phichallenge/dataset/3ddwa5567'
    zip_path = 'data.zip'

    # Download the dataset
    print(f"📥 Downloading from: {url}")
    urllib.request.urlretrieve(url, zip_path)
    print(f"✅ Downloaded to {zip_path}")

    # Extract the dataset
    print("📂 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('.')

    print("✅ Dataset downloaded and extracted successfully!")
    return 'task5'

# Only run if using PEER Hub method
# (Will be skipped if dataset already downloaded via Hugging Face)
if not os.path.exists('task5'):
    print("Note: If you ran the Hugging Face download above, you can skip this cell.")
    download_dataset()
else:
    print("✅ Dataset already exists!")

In [ ]:
def preprocess_phinet_data(datadir='task5/', output_dir='processed_data'):
    """
    Preprocess Φ-Net Data into PyTorch ImageFolder format
    """
    classes = ['GC', 'NC', 'PC']

    print(f"Processing Φ-Net data from {datadir}...")

    for phase in ['train', 'val']:
        # Create directories for each class
        for cl in classes:
            os.makedirs(f'{output_dir}/{phase}/{cl}', exist_ok=True)

        # Map phase names
        if phase == 'val':
            phasew = 'test'
        else:
            phasew = 'train'

        # Load numpy arrays
        print(f"Loading {phase} data...")
        ims = np.load(f'{datadir}/task5_X_{phasew}.npy')
        labels = np.load(f'{datadir}/task5_y_{phasew}.npy')

        # Normalize pixel values
        maxpix = np.amax(ims)
        minpix = np.amin(ims)

        # Process labels
        labels[:, 1] *= 2
        labels[:, 2] *= 3
        labels = (labels[np.where(labels != 0)] - 1).astype(int)

        # Convert and save images
        print(f"Converting {len(ims)} {phase} images...")
        for im_idx in tqdm(range(len(ims)), desc=f"Processing {phase}"):
            img = ((ims[im_idx, :, :, :]) - minpix) * 255 / (maxpix - minpix)
            img_rgb = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2RGB)
            cl = classes[labels[im_idx]]
            output_path = f'{output_dir}/{phase}/{cl}/{im_idx}.jpg'
            cv2.imwrite(output_path, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))

        print(f"Processed {phase} data successfully!")

    # Print dataset statistics
    print("\nDataset Statistics:")
    for phase in ['train', 'val']:
        print(f"\n{phase.upper()} SET:")
        for cl in classes:
            count = len(os.listdir(f'{output_dir}/{phase}/{cl}'))
            print(f"  {cl}: {count} images")

# Preprocess the data
if not os.path.exists('processed_data'):
    preprocess_phinet_data(datadir='task5/', output_dir='processed_data')
else:
    print("Processed data already exists!")
    # Show statistics
    classes = ['GC', 'NC', 'PC']
    for phase in ['train', 'val']:
        print(f"\n{phase.upper()} SET:")
        for cl in classes:
            count = len(os.listdir(f'processed_data/{phase}/{cl}'))
            print(f"  {cl}: {count} images")

In [ ]:
def create_balanced_dataset(input_dir='processed_data', output_dir='processed_data_balanced'):
    """Create a balanced training dataset"""
    classes = ['GC', 'NC', 'PC']

    print("Creating balanced dataset...")

    # Find minimum samples
    class_counts = {}
    for cl in classes:
        class_counts[cl] = len(os.listdir(f'{input_dir}/train/{cl}'))

    min_samples = min(class_counts.values())
    print(f"Original class distribution: {class_counts}")
    print(f"Balancing to {min_samples} samples per class...")

    # Create balanced training set
    for cl in classes:
        dir_path = f'{output_dir}/train/{cl}'
        os.makedirs(dir_path, exist_ok=True)

        ims = os.listdir(f'{input_dir}/train/{cl}')
        ims_sampled = random.sample(ims, min_samples)

        for im in ims_sampled:
            shutil.copyfile(
                f'{input_dir}/train/{cl}/{im}',
                f'{output_dir}/train/{cl}/{im}'
            )

    # Copy validation set
    if os.path.exists(f'{output_dir}/val'):
        shutil.rmtree(f'{output_dir}/val')
    shutil.copytree(f'{input_dir}/val', f'{output_dir}/val')

    print("\nBalanced Dataset Statistics:")
    for phase in ['train', 'val']:
        print(f"\n{phase.upper()} SET:")
        for cl in classes:
            count = len(os.listdir(f'{output_dir}/{phase}/{cl}'))
            print(f"  {cl}: {count} images")

# Create balanced dataset
if not os.path.exists('processed_data_balanced'):
    create_balanced_dataset('processed_data', 'processed_data_balanced')
else:
    print("Balanced dataset already exists!")

## 3. Visualize Sample Images

In [ ]:
def visualize_samples(data_dir='processed_data_balanced', n_samples=3):
    """Visualize sample images from each class"""
    classes = ['GC', 'NC', 'PC']
    class_names = {
        'GC': 'Global Collapse',
        'NC': 'No Collapse',
        'PC': 'Partial Collapse'
    }

    fig, axes = plt.subplots(len(classes), n_samples, figsize=(14, 10))

    for i, cl in enumerate(classes):
        class_dir = f'{data_dir}/train/{cl}'
        images = os.listdir(class_dir)
        samples = random.sample(images, n_samples)

        for j, img_name in enumerate(samples):
            img_path = os.path.join(class_dir, img_name)
            img = plt.imread(img_path)

            axes[i, j].imshow(img)
            axes[i, j].axis('off')

            # Add class label (GC, NC, PC) as title for each column
            axes[i, j].set_title(f'{cl}', fontsize=13, fontweight='bold', pad=10)

            # Add full class name on the left
            if j == 0:
                axes[i, j].set_ylabel(class_names[cl], fontsize=13, fontweight='bold', rotation=0,
                                     labelpad=80, ha='right', va='center')

    plt.suptitle('Sample Images from Each Damage Class', fontsize=16, fontweight='bold', y=0.98)

    # Add a text box with class descriptions
    description_text = (
        'GC (Global Collapse): Complete structural failure  |  '
        'NC (No Collapse): No visible structural damage  |  '
        'PC (Partial Collapse): Partial structural damage'
    )
    fig.text(0.5, 0.01, description_text, ha='center', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_samples()

## 4. Create PyTorch Model and Training Setup

In [ ]:
class BuildingDamageClassifier:
    def __init__(self, model_name='resnet50', num_classes=3, learning_rate=0.001):
        # Auto-detect best device: CUDA > MPS > CPU



        # Initialize model


        # Loss function


        # Training history
        self.history = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': []
        }

    def _create_model(self, model_name, num_classes):
        """Create and configure the model"""

        # ResNet Models
        if model_name == 'resnet50':
            model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
            num_ftrs = model.fc.in_features
            model.fc = nn.Linear(num_ftrs, num_classes)

        elif model_name == 'resnet18':
            model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            num_ftrs = model.fc.in_features
            model.fc = nn.Linear(num_ftrs, num_classes)

        elif model_name == 'resnet101':
            model = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
            num_ftrs = model.fc.in_features
            model.fc = nn.Linear(num_ftrs, num_classes)

        # EfficientNet Models (RECOMMENDED)
        elif model_name == 'efficientnet_v2_s':
            model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
            num_ftrs = model.classifier[1].in_features
            model.classifier[1] = nn.Linear(num_ftrs, num_classes)

        elif model_name == 'efficientnet_v2_m':
            model = models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.IMAGENET1K_V1)
            num_ftrs = model.classifier[1].in_features
            model.classifier[1] = nn.Linear(num_ftrs, num_classes)

        elif model_name == 'efficientnet_v2_l':
            model = models.efficientnet_v2_l(weights=models.EfficientNet_V2_L_Weights.IMAGENET1K_V1)
            num_ftrs = model.classifier[1].in_features
            model.classifier[1] = nn.Linear(num_ftrs, num_classes)

        # ConvNeXt Models (STATE-OF-THE-ART)
        elif model_name == 'convnext_tiny':
            model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
            num_ftrs = model.classifier[2].in_features
            model.classifier[2] = nn.Linear(num_ftrs, num_classes)

        elif model_name == 'convnext_small':
            model = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
            num_ftrs = model.classifier[2].in_features
            model.classifier[2] = nn.Linear(num_ftrs, num_classes)

        elif model_name == 'convnext_base':
            model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
            num_ftrs = model.classifier[2].in_features
            model.classifier[2] = nn.Linear(num_ftrs, num_classes)

        # Vision Transformer (ViT)
        elif model_name == 'vit_b_16':
            model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
            num_ftrs = model.heads.head.in_features
            model.heads.head = nn.Linear(num_ftrs, num_classes)

        elif model_name == 'vit_b_32':
            model = models.vit_b_32(weights=models.ViT_B_32_Weights.IMAGENET1K_V1)
            num_ftrs = model.heads.head.in_features
            model.heads.head = nn.Linear(num_ftrs, num_classes)

        # RegNet Models
        elif model_name == 'regnet_y_8gf':
            model = models.regnet_y_8gf(weights=models.RegNet_Y_8GF_Weights.IMAGENET1K_V2)
            num_ftrs = model.fc.in_features
            model.fc = nn.Linear(num_ftrs, num_classes)

        else:
            raise ValueError(f"Unknown model: {model_name}. Available models: "
                           "resnet18, resnet50, resnet101, "
                           "efficientnet_v2_s, efficientnet_v2_m, efficientnet_v2_l, "
                           "convnext_tiny, convnext_small, convnext_base, "
                           "vit_b_16, vit_b_32, regnet_y_8gf")

        print(f"Created {model_name} model with {num_classes} output classes")

        # Print parameter count
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")

        return model

    def prepare_data(self, data_dir, batch_size=32, image_size=224):
        """Prepare data loaders"""
        data_transforms = {
            'train': transforms.Compose([
                transforms.RandomResizedCrop(image_size),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ]),
            'val': transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(image_size),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ]),
        }

        # Load datasets
        image_datasets = {
            x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
            for x in ['train', 'val']
        }

        # Create data loaders
        # Note: Use num_workers=0 for MPS to avoid multiprocessing issues
        num_workers = 0 if self.device.type == 'mps' else 4

        self.dataloaders = {
            x: DataLoader(image_datasets[x], batch_size=batch_size,
                         shuffle=True, num_workers=num_workers)
            for x in ['train', 'val']
        }

        self.dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
        self.class_names = image_datasets['train'].classes

        print(f"Dataset loaded:")
        print(f"  Classes: {self.class_names}")
        print(f"  Training samples: {self.dataset_sizes['train']}")
        print(f"  Validation samples: {self.dataset_sizes['val']}")

    def _freeze_backbone(self):
        """Freeze all layers except the final classifier"""
        for param in self.model.parameters():
            param.requires_grad = False

        if 'resnet' in self.model_name or 'regnet' in self.model_name:
            for param in self.model.fc.parameters():
                param.requires_grad = True
        elif 'efficientnet' in self.model_name:
            for param in self.model.classifier.parameters():
                param.requires_grad = True
        elif 'convnext' in self.model_name:
            for param in self.model.classifier.parameters():
                param.requires_grad = True
        elif 'vit' in self.model_name:
            for param in self.model.heads.parameters():
                param.requires_grad = True

    def _unfreeze_backbone(self):
        """Unfreeze all layers for fine-tuning"""
        for param in self.model.parameters():
            param.requires_grad = True

    def train_epoch(self, optimizer, phase):
        """Train or validate for one epoch"""
        if phase == 'train':
            self.model.train()
        else:
            self.model.eval()

        running_loss = 0.0
        running_corrects = 0

        pbar = tqdm(self.dataloaders[phase], desc=f'{phase.capitalize()}')
        for inputs, labels in pbar:
            inputs = inputs.to(self.device)
            labels = labels.to(self.device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = self.model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = self.criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

            pbar.set_postfix({'loss': loss.item()})

        epoch_loss = running_loss / self.dataset_sizes[phase]

        # Fix for MPS: Use float() instead of double() to avoid float64 conversion
        if self.device.type == 'mps':
            epoch_acc = running_corrects.float() / self.dataset_sizes[phase]
        else:
            epoch_acc = running_corrects.double() / self.dataset_sizes[phase]

        return epoch_loss, epoch_acc

    def train(self, num_epochs=15, freeze_backbone=True, unfreeze_after=3):
        """Train the model"""
        since = time.time()
        best_model_wts = copy.deepcopy(self.model.state_dict())
        best_acc = 0.0

        # Phase 1: Train classifier head only
        if freeze_backbone:
            print(f"\nPhase 1: Training classifier head ({unfreeze_after} epochs)...")
            self._freeze_backbone()
            optimizer = optim.Adam(
                filter(lambda p: p.requires_grad, self.model.parameters()),
                lr=self.learning_rate
            )
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

            for epoch in range(unfreeze_after):
                print(f'\nEpoch {epoch}/{unfreeze_after-1}')
                print('-' * 60)

                for phase in ['train', 'val']:
                    epoch_loss, epoch_acc = self.train_epoch(optimizer, phase)

                    print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

                    self.history[f'{phase}_loss'].append(epoch_loss)
                    self.history[f'{phase}_acc'].append(epoch_acc.item())

                    if phase == 'val' and epoch_acc > best_acc:
                        best_acc = epoch_acc
                        best_model_wts = copy.deepcopy(self.model.state_dict())

                if phase == 'train':
                    scheduler.step()

            # Phase 2: Fine-tune entire model
            print(f"\nPhase 2: Fine-tuning entire model ({num_epochs - unfreeze_after} epochs)...")
            self._unfreeze_backbone()

        # Continue training
        optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate / 10)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

        start_epoch = unfreeze_after if freeze_backbone else 0
        for epoch in range(start_epoch, num_epochs):
            print(f'\nEpoch {epoch}/{num_epochs-1}')
            print('-' * 60)

            for phase in ['train', 'val']:
                epoch_loss, epoch_acc = self.train_epoch(optimizer, phase)

                print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

                self.history[f'{phase}_loss'].append(epoch_loss)
                self.history[f'{phase}_acc'].append(epoch_acc.item())

                if phase == 'val' and epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(self.model.state_dict())

            scheduler.step()

        time_elapsed = time.time() - since
        print(f'\nTraining complete in {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s')
        print(f'Best val Acc: {best_acc:.4f}')

        self.model.load_state_dict(best_model_wts)

    def save_model(self, path='models/best_model.pth'):
        """Save the trained model"""
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'model_name': self.model_name,
            'num_classes': self.num_classes,
            'class_names': self.class_names,
            'history': self.history
        }, path)
        print(f"Model saved to {path}")

    def plot_history(self):
        """Plot training history"""
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Plot accuracy
        axes[0].plot(self.history['train_acc'], label='Train', marker='o')
        axes[0].plot(self.history['val_acc'], label='Validation', marker='s')
        axes[0].set_title('Model Accuracy', fontweight='bold')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Plot loss
        axes[1].plot(self.history['train_loss'], label='Train', marker='o')
        axes[1].plot(self.history['val_loss'], label='Validation', marker='s')
        axes[1].set_title('Model Loss', fontweight='bold')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Training history saved to training_history.png")

## 5. Train the Model

**Model Selection:** Based on our comparison study of 5 top models, **ConvNeXt-Small** achieved the best accuracy (77.40%) and is now set as the default.

You can also try other models by changing the `model_name` parameter:
- `convnext_small` - **BEST** (77.40% accuracy) ⭐
- `efficientnet_v2_s` - BRAILS model (76.71% accuracy)
- `resnet50` - Baseline (73.97% accuracy)
- `efficientnet_v2_m` - (71.92% accuracy)

In [ ]:
# Create classifier
# 🌟 MODEL COMPARISON RESULTS (10 epochs, 966 train images)
#
# We trained and compared the top 5 models. Here are the results:
#
# ╔═══════════════════════╦═════════════════╦══════════════╗
# ║ Model                 ║ Best Val Acc    ║ Training Time║
# ╠═══════════════════════╬═════════════════╬══════════════╣
# ║ convnext_small        ║ 77.40% ⭐ BEST ║ 2m 25s       ║
# ║ efficientnet_v2_s     ║ 76.71%         ║ 2m 36s       ║
# ║ resnet50              ║ 73.97%         ║ 2m 7s        ║
# ║ efficientnet_v2_m     ║ 71.92%         ║ 2m 42s       ║
# ╚═══════════════════════╩═════════════════╩══════════════╝
#
# 🏆 WINNER: ConvNeXt-Small (77.40% accuracy)
#    - State-of-the-art CNN architecture (2022)
#    - 5.4% better than ResNet50
#    - Only 18 seconds slower than ResNet50
#
# 💡 RECOMMENDATION: Use 'convnext_small' for best accuracy!

classifier = BuildingDamageClassifier(
    model_name='convnext_small',  # ⭐ BEST: 77.40% accuracy (proven in comparison)
    num_classes=3,
    learning_rate=0.001
)

# Other options to try:
# model_name='efficientnet_v2_s'  # 76.71% - What BRAILS uses, fast and accurate
# model_name='resnet50'           # 73.97% - Original baseline
# model_name='efficientnet_v2_m'  # 71.92% - Surprisingly underperformed

In [ ]:
# Prepare data
classifier.prepare_data(
    data_dir='processed_data_balanced',
    batch_size=32
)

In [ ]:
# Train the model
# This will take some time depending on your hardware
# CPU: ~30-45 minutes for 10 epochs
# GPU: ~5-10 minutes for 10 epochs

classifier.train(num_epochs=20, freeze_backbone=True, unfreeze_after=3)

In [ ]:
# Save the trained model
classifier.save_model('models/best_model.pth')

In [ ]:
# Plot training history
classifier.plot_history()

## 6. Evaluate the Model

In [ ]:
def evaluate_model(classifier, data_dir='processed_data_balanced', phase='val'):
    """Evaluate model on validation set"""
    classifier.model.eval()

    y_true = []
    y_pred = []
    y_probs = []

    eval_dir = os.path.join(data_dir, phase)

    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    print(f"Evaluating on {eval_dir}...")

    for class_name in classifier.class_names:
        class_dir = os.path.join(eval_dir, class_name)
        if not os.path.exists(class_dir):
            continue

        image_files = [f for f in os.listdir(class_dir)
                      if f.endswith(('.jpg', '.jpeg', '.png'))]

        print(f"Evaluating {class_name}: {len(image_files)} images")

        for img_file in tqdm(image_files, desc=f"Processing {class_name}"):
            img_path = os.path.join(class_dir, img_file)
            image = Image.open(img_path).convert('RGB')
            input_tensor = transform(image).unsqueeze(0).to(classifier.device)

            with torch.no_grad():
                outputs = classifier.model(input_tensor)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs, 1)

            y_true.append(class_name)
            y_pred.append(classifier.class_names[predicted.item()])
            y_probs.append(probs[0].cpu().numpy())

    # Print classification report
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(y_true, y_pred, target_names=classifier.class_names))

    # Plot confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=classifier.class_names)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=classifier.class_names,
               yticklabels=classifier.class_names,
               cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix', fontweight='bold', fontsize=14)
    plt.ylabel('True Label', fontweight='bold')
    plt.xlabel('Predicted Label', fontweight='bold')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Calculate per-class accuracy
    print("\n" + "="*60)
    print("PER-CLASS ACCURACY")
    print("="*60)
    for i, class_name in enumerate(classifier.class_names):
        class_acc = cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0
        print(f"{class_name}: {class_acc:.4f} ({class_acc*100:.2f}%)")

    overall_acc = np.trace(cm) / np.sum(cm)
    print(f"\nOverall Accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)")

    return y_true, y_pred, y_probs

# Evaluate the model
y_true, y_pred, y_probs = evaluate_model(classifier, 'processed_data_balanced', 'val')

## 7. Visualize Predictions

In [ ]:
def visualize_predictions(classifier, data_dir='processed_data_balanced', n_samples=3):
    """Visualize predictions for sample images"""
    classifier.model.eval()

    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    fig, axes = plt.subplots(len(classifier.class_names), n_samples, figsize=(12, 9))

    for i, class_name in enumerate(classifier.class_names):
        class_dir = f'{data_dir}/val/{class_name}'
        images = os.listdir(class_dir)
        samples = random.sample(images, n_samples)

        for j, img_name in enumerate(samples):
            img_path = os.path.join(class_dir, img_name)

            # Load and display image
            img_display = plt.imread(img_path)

            # Make prediction
            image = Image.open(img_path).convert('RGB')
            input_tensor = transform(image).unsqueeze(0).to(classifier.device)

            with torch.no_grad():
                outputs = classifier.model(input_tensor)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs, 1)

            pred_class = classifier.class_names[predicted.item()]
            confidence = probs[0][predicted.item()].item()

            # Display
            axes[i, j].imshow(img_display)
            axes[i, j].axis('off')

            # Color code: green if correct, red if wrong
            color = 'green' if pred_class == class_name else 'red'
            axes[i, j].set_title(f'Pred: {pred_class}\n({confidence:.2f})',
                                color=color, fontsize=9, fontweight='bold')

            if j == 0:
                axes[i, j].set_ylabel(f'True: {class_name}',
                                     fontsize=11, fontweight='bold')

    plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)',
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('prediction_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_predictions(classifier, 'processed_data_balanced', n_samples=3)

## 8. Make Predictions on New Images

In [ ]:
def predict_single_image(classifier, image_path):
    """Predict damage class for a single image"""
    classifier.model.eval()

    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Load image
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(classifier.device)

    # Make prediction
    with torch.no_grad():
        outputs = classifier.model(input_tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)

    pred_class = classifier.class_names[predicted.item()]
    probabilities = {classifier.class_names[i]: probs[0][i].item()
                    for i in range(len(classifier.class_names))}

    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Show image
    ax1.imshow(image)
    ax1.set_title(f'Predicted: {pred_class}', fontweight='bold', fontsize=12)
    ax1.axis('off')

    # Show probabilities
    classes = list(probabilities.keys())
    probs_list = list(probabilities.values())
    colors = ['green' if cls == pred_class else 'gray' for cls in classes]

    ax2.barh(classes, probs_list, color=colors)
    ax2.set_xlabel('Probability', fontweight='bold')
    ax2.set_title('Class Probabilities', fontweight='bold', fontsize=12)
    ax2.set_xlim([0, 1])

    for i, (cls, prob) in enumerate(zip(classes, probs_list)):
        ax2.text(prob + 0.02, i, f'{prob:.3f}', va='center')

    plt.tight_layout()
    plt.show()

    print(f"\nPredicted class: {pred_class}")
    print("\nClass probabilities:")
    for cls, prob in probabilities.items():
        print(f"  {cls}: {prob:.4f} ({prob*100:.2f}%)")

    return pred_class, probabilities

# Example: predict on a random validation image
sample_class = random.choice(classifier.class_names)
sample_dir = f'processed_data_balanced/val/{sample_class}'
sample_image = random.choice(os.listdir(sample_dir))
sample_path = os.path.join(sample_dir, sample_image)

print(f"Testing on a sample image from class: {sample_class}")
pred_class, probs = predict_single_image(classifier, sample_path)